In [1]:
import torch
import numpy as np
print(torch.cuda.is_available())
import os
os.environ["CUDA_VISIBLE_DEVICES"]="0"
print(torch.cuda.get_device_name(0))

True
NVIDIA A100-SXM4-40GB MIG 3g.20gb


In [2]:
shape1 = torch.randn(2, 3, 5, 8)
shape2 = torch.randn(2, 3, 5, 4)
shape3 = torch.concat((shape1, shape2), dim=3)
print(shape3.shape)

torch.Size([2, 3, 5, 12])


In [2]:
batch_size = 8
channels = 2
timesteps = 16 
height = 128
width = 128
shape = (batch_size, channels, timesteps, height, width)

amount = 10

In [3]:
for i in enumerate(range(amount)):
    gaussian_noise_np = np.random.normal(size=shape)
print(gaussian_noise_np.shape, end='\r')


In [4]:
for i in enumerate(range(amount)):
    gaussian_noise_torch = torch.randn(batch_size, channels, timesteps, height, width)
    print(i, gaussian_noise_torch.shape, end='\r')

print(gaussian_noise_torch.shape)  # Output: torch.Size([4, 3, 64, 64])


torch.Size([8, 2, 16, 128, 128]), 128])


In [8]:
import sys
import os
import torch
sys.path.append('../../flow_matching/')
print(sys.path[-1])
print(os.listdir(sys.path[-1]))
from examples.image.models.unet import UNetModel
model = UNetModel(in_channels = 2,
        model_channels = 48,
        out_channels = 2,
        num_res_blocks = 1,
        attention_resolutions = [0,0,8],#[2, 4, 8],
        dropout = 0.1,
        channel_mult = [1, 2, 2, 2],
        num_classes = None,
        use_checkpoint = False,
        num_heads = 4,
        num_head_channels = 48,
        use_scale_shift_norm = True,
        resblock_updown = True,
        use_new_attention_order = True,
        with_fourier_features = False,
        dims = 3
)
checkpoint = torch.load('../../output/1709-14hr/FluidGPT_FM/1k592osu/epoch=0011-val_SS_loss_checkpoint=0.009457.ckpt', map_location='cpu')
new_state_dict = {k.replace('model.', ''): v for k, v in checkpoint['state_dict'].items()}
model.load_state_dict(new_state_dict)

../../flow_matching/
['setup.py', 'tests', 'docs', '.pre-commit-config.yaml', 'LICENSE', 'CONTRIBUTING.md', '.github', 'environment.yml', 'assets', '.flake8', 'flow_matching', 'examples', 'README.md', '.gitignore', 'CHANGELOG.md', 'CODE_OF_CONDUCT.md', '.git', 'RELEASE.md']


<All keys matched successfully>

In [2]:
import sys
sys.path.append('../')
%load_ext autoreload
%autoreload
from modelComp.FluidGPT_FM import FluidGPT_FM
from modelComp.utils import ACT_MAPPER, SKIPBLOCK_MAPPER
model = FluidGPT_FM(
    data_dim=[1, 16, 2, 128, 128],
    emb_dim=96,
    patch_size=(8, 8),
    hiddenout_dim=256,
    flowmatching_emb_dim=256,
    depth=2,
    stage_depths=[12,12,12,12,12],
    num_heads=[32,32,64,32,32],
    window_size=4,
    use_flex_attn=True,
    act=ACT_MAPPER['gelu'],
    skip_connect=SKIPBLOCK_MAPPER['convnext'],
    #norm_layer=torch.nn.GroupNorm,
    gradient_flowthrough=[True, True, True]
).to('cuda')

/usr/local/lib/python3.11/dist-packages/torch/functional.py:554: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4314.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [3]:
shape = (2, 2, 16, 128, 128)
xtest = torch.randn(shape)
xtest = xtest.to('cuda')
print(xtest.shape)
t = torch.rand((2), device='cuda')
print(t)
y = model(xtest, t)
print(y.mean(), y.std(), y.shape)

torch.Size([2, 2, 16, 128, 128])
tensor([0.1138, 0.0443], device='cuda:0')
tensor(0.0003, device='cuda:0', grad_fn=<MeanBackward0>) tensor(0.0788, device='cuda:0', grad_fn=<StdBackward0>) torch.Size([2, 2, 16, 128, 128])


In [5]:
model.resnorms_up

ModuleList(
  (0): ModuleList(
    (0-11): 12 x LayerNorm((192,), eps=1e-05, elementwise_affine=True)
  )
  (1): ModuleList(
    (0-11): 12 x LayerNorm((96,), eps=1e-05, elementwise_affine=True)
  )
)